# MuSeg-AI Thigh Segmentation

Runs [fabianbalsiger/museg-ai](https://github.com/fabianbalsiger/museg-ai) (nnU-Net `thigh-model3`)
on fat-fraction stacks.

The model expects two Dixon volumes (in-phase + out-of-phase). Since only fat-fraction
images are available here, the same volume is passed as both channels â€” a known approximation.

**Requirements:**
- Docker must be running (the museg-ai package calls Docker internally)
- First run will pull `fabianbalsiger/museg:thigh-model3` (~several GB)

Output: `museg_thigh_segs/*_museg.nii.gz` in original image space.

**Kernel:** `dafne_clean`

In [10]:
import importlib
if importlib.util.find_spec("musegai") is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "git+https://github.com/fabianbalsiger/museg-ai.git"])
import musegai
print("museg-ai version:", musegai.__version__)

museg-ai version: 0.1.0


In [11]:
# Verify Docker is reachable before starting
import docker
try:
    client = docker.from_env()
    client.ping()
    print('Docker is running')
except Exception as e:
    raise RuntimeError(
        'Docker is not reachable. Please start Docker Desktop and re-run this cell.'
    ) from e

Docker is running


In [12]:
import glob
import os
import numpy as np
from musegai import api

EVAL_DIR   = r'C:\Projects\dissector\eval_notebooks'
IMAGE_GLOB = os.path.join(EVAL_DIR, 'myosegmenTUM', '*', 'ImageData',
                          '*FATFRACTION', '*FATFRACTION_stack*.nii')
OUTPUT_DIR = os.path.join(EVAL_DIR, 'museg_thigh_segs')

LABEL_MAP = {
    1:  'Vastus_Lateralis',
    2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',
    4:  'Rectus_Femoris',
    5:  'Sartorius',
    6:  'Gracilis',
    7:  'Semimembranosus',
    8:  'Semitendinosus',
    9:  'Biceps_Femoris',
    10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',
    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} fat-fraction stacks')

Found 54 fat-fraction stacks


In [13]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_museg.nii.gz")

    if os.path.exists(out_path):
        print(f"Skipping (done): {stem}")
        continue

    print(f"\nProcessing: {stem}")
    vol = api.Volume.load(nii_path)
    print(f"  Shape: {vol.shape}  Spacing: {vol.spacing}")

    # API expects {name: [Volume_ch0, Volume_ch1]}
    # Pass fat-fraction as both channels (approximation for missing Dixon pair)
    results, labels = api.segment_volumes(
        {stem: [vol, vol]},
        model="thigh-model3",
        side="left+right",
    )

    segmentation = results[stem]
    segmentation.save(out_path)
    print(f"  Saved -> {out_path}")

    seg_arr = segmentation.array
    print(f"  Labels present: {sorted(np.unique(seg_arr).tolist())}")
    print(f"  {'Label':<6} {'Muscle':<25} {'Voxels':>10}")
    print(f"  {'-'*45}")
    for idx, name in LABEL_MAP.items():
        n = int((seg_arr == idx).sum())
        if n > 0:
            print(f"  {idx:<6} {name:<25} {n:>10,}")

print("\nAll done.")


Processing: HV001_1_FATFRACTION_stack1
  Shape: (672, 672, 65)  Spacing: (1.0, 1.0, 4.0)
Pulling image `fabianbalsiger/museg:thigh-model3`, this may take a while...
Running inference model 'thigh-model3' (`fabianbalsiger/museg:thigh-model3`)


APIError: 500 Server Error for http+docker://localnpipe/v1.54/containers/00a7965f106941b9a94cba38d7715607a04593d8437765a904f164d31dd0749a/start: Internal Server Error ("failed to create task for container: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: error during container init: error running prestart hook #0: exit status 1, stdout: , stderr: Auto-detected mode as 'legacy'
nvidia-container-cli: initialization error: WSL environment detected but no adapters were found")

In [ ]:
# Sanity check
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.nii.gz')))
if results:
    sample = api.Volume.load(results[0])
    print('Sample:', results[0])
    print('  Shape:',   sample.shape)
    print('  Spacing:', sample.spacing)
    print('  Labels:', sorted(np.unique(sample.array).tolist()))
else:
    print('No results yet.')